# EEG Pipeline for Emognition Dataset (5 Emotion Classes)

This notebook recreates the EEG processing pipeline for the **Emognition dataset** using a BIH-GCN (Brain-Inspired Hierarchical Graph Convolutional Network) architecture for 5-class emotion classification.

## Pipeline Overview
1. Load and prepare raw EEG data
2. Segment EEG into fixed windows
3. Preprocess EEG data (filtering + normalization)
4. Convert EEG channels to spectrograms via STFT
5. Build CNN-based EEG encoder
6. Map channels to brain regions
7. Construct BIH-GCN graph
8. Train the model
9. Evaluate the model
10. Standalone testing script

In [ ]:
# ─────────────────────────────────────────────
# Install / Import Required Libraries
# ─────────────────────────────────────────────
import os
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from scipy.signal import butter, filtfilt, iirnotch
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")

# ── Reproducibility ──────────────────────────
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Constants ────────────────────────────────
SAMPLE_RATE    = 128          # Hz
N_CHANNELS     = 4            # EEG channels
WINDOW_SAMPLES = 512          # ~4 seconds @ 128 Hz
N_CLASSES      = 5            # 5 emotion classes
EMOTION_LABELS = {0: "Neutral", 1: "Happy", 2: "Sad", 3: "Angry", 4: "Fear"}

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")
print(f"PyTorch version: {torch.__version__}")

---
## Section 1 — Load and Prepare Raw EEG Data

Load the raw EEG data shaped as `(samples, 4 channels, time)` and prepare it for processing.  
In a real scenario you would replace the synthetic generator below with your actual data loader.

In [ ]:
# ─────────────────────────────────────────────
# 1. Load and Prepare Raw EEG Data
# ─────────────────────────────────────────────

def load_raw_eeg(data_path: str = None):
    """
    Load raw EEG data from disk OR generate synthetic data for demonstration.

    Expected shape: (N_samples, N_channels, T_timepoints)
    Labels shape  : (N_samples,)  — integer in [0, N_CLASSES-1]
    """
    if data_path and os.path.exists(data_path):
        data   = np.load(os.path.join(data_path, "eeg_data.npy"))
        labels = np.load(os.path.join(data_path, "labels.npy"))
        print(f"Loaded real data  — shape: {data.shape}, labels: {labels.shape}")
    else:
        print("⚠  No data path provided — generating SYNTHETIC EEG data for demo.")
        N         = 500          # number of trials
        T         = 1024         # time-points per trial (≈ 8 s)
        data      = np.random.randn(N, N_CHANNELS, T).astype(np.float32)
        labels    = np.random.randint(0, N_CLASSES, size=(N,)).astype(np.int64)
        print(f"Synthetic data — shape: {data.shape}, labels: {labels.shape}")

    return data, labels


# ── Channel metadata ─────────────────────────
CHANNEL_NAMES  = ["Fp1", "Fp2", "T7", "T8"]          # 4 channels
CHANNEL_REGION = {"Fp1": "Frontal", "Fp2": "Frontal",
                  "T7":  "Temporal","T8":  "Temporal"}

raw_data, raw_labels = load_raw_eeg(data_path=None)

# ── Quick sanity check ───────────────────────
print(f"\nData  shape : {raw_data.shape}   (trials × channels × time)")
print(f"Label shape : {raw_labels.shape}")
print(f"Label dist  : { {k: int((raw_labels==k).sum()) for k in range(N_CLASSES)} }")

# ── Visualise one trial ──────────────────────
fig, axes = plt.subplots(N_CHANNELS, 1, figsize=(12, 6), sharex=True)
t_axis = np.arange(raw_data.shape[-1]) / SAMPLE_RATE
for i, ax in enumerate(axes):
    ax.plot(t_axis, raw_data[0, i], lw=0.7)
    ax.set_ylabel(CHANNEL_NAMES[i], fontsize=9)
axes[-1].set_xlabel("Time (s)")
fig.suptitle("Raw EEG — Trial 0", fontsize=12)
plt.tight_layout()
plt.show()

---
## Section 2 — Segment EEG Data into Fixed Windows

Segment each trial into non-overlapping windows of **512 samples (~4 s @ 128 Hz)**.

In [ ]:
# ─────────────────────────────────────────────
# 2. Segment EEG Data into Fixed Windows
# ─────────────────────────────────────────────

def segment_eeg(data: np.ndarray,
                labels: np.ndarray,
                window_size: int = WINDOW_SAMPLES,
                overlap: float = 0.0) -> tuple:
    """
    Slide a window over the time axis of each trial.

    Parameters
    ----------
    data        : (N, C, T)
    labels      : (N,)
    window_size : number of samples per window
    overlap     : fraction of overlap between consecutive windows [0, 1)

    Returns
    -------
    segments : (M, C, window_size)   — M = total windows across all trials
    seg_labels: (M,)
    """
    step = int(window_size * (1.0 - overlap))
    segments, seg_labels = [], []

    for trial_idx in range(data.shape[0]):
        T = data.shape[-1]
        start = 0
        while start + window_size <= T:
            seg = data[trial_idx, :, start:start + window_size]
            segments.append(seg)
            seg_labels.append(labels[trial_idx])
            start += step

    segments   = np.stack(segments,   axis=0).astype(np.float32)
    seg_labels = np.array(seg_labels, dtype=np.int64)
    return segments, seg_labels


segments, seg_labels = segment_eeg(raw_data, raw_labels,
                                   window_size=WINDOW_SAMPLES,
                                   overlap=0.0)

print(f"Segments shape : {segments.shape}   (windows × channels × time)")
print(f"Labels   shape : {seg_labels.shape}")
print(f"Window duration: {WINDOW_SAMPLES / SAMPLE_RATE:.1f} s")

---
## Section 3 — Preprocess EEG Data

Apply:
- **1–50 Hz Butterworth bandpass filter** (4th order)
- **50 Hz notch filter** (Q = 30)
- **Per-channel z-score normalization** (zero mean, unit variance)

In [ ]:
# ─────────────────────────────────────────────
# 3. Preprocess EEG Data
# ─────────────────────────────────────────────

def bandpass_filter(data: np.ndarray,
                    lowcut: float = 1.0,
                    highcut: float = 50.0,
                    fs: float = SAMPLE_RATE,
                    order: int = 4) -> np.ndarray:
    """Apply a zero-phase Butterworth bandpass filter."""
    nyq  = 0.5 * fs
    b, a = butter(order, [lowcut / nyq, highcut / nyq], btype="band")
    return filtfilt(b, a, data, axis=-1).astype(np.float32)


def notch_filter(data: np.ndarray,
                 notch_freq: float = 50.0,
                 fs: float = SAMPLE_RATE,
                 quality: float = 30.0) -> np.ndarray:
    """Apply a zero-phase IIR notch filter."""
    b, a = iirnotch(notch_freq / (0.5 * fs), quality)
    return filtfilt(b, a, data, axis=-1).astype(np.float32)


def normalize_per_channel(data: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """Z-score each channel independently across its time axis."""
    mu  = data.mean(axis=-1, keepdims=True)
    std = data.std( axis=-1, keepdims=True) + eps
    return ((data - mu) / std).astype(np.float32)


def preprocess_eeg(data: np.ndarray) -> np.ndarray:
    """Full preprocessing pipeline for (M, C, T) array."""
    print("  → Bandpass filtering  [1–50 Hz] …", end=" ")
    data = bandpass_filter(data)
    print("done")

    print("  → Notch filtering     [50 Hz]   …", end=" ")
    data = notch_filter(data)
    print("done")

    print("  → Per-channel z-score …",           end=" ")
    data = normalize_per_channel(data)
    print("done")

    return data


print("Preprocessing segments …")
preprocessed = preprocess_eeg(segments.copy())

print(f"\nPreprocessed shape : {preprocessed.shape}")
print(f"Mean (should ≈ 0) : {preprocessed.mean():.4f}")
print(f"Std  (should ≈ 1) : {preprocessed.std():.4f}")

# ── Visualise before vs after ────────────────
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(segments[0, 0],      lw=0.7, label="Raw");         axes[0].set_title("Before preprocessing"); axes[0].legend()
axes[1].plot(preprocessed[0, 0],  lw=0.7, label="Preprocessed",color="C1"); axes[1].set_title("After preprocessing"); axes[1].legend()
plt.tight_layout(); plt.show()

---
## Section 4 — Convert EEG Channels to Spectrograms

Use **Short-Time Fourier Transform (STFT)** with:
- `n_fft = 256`
- `hop_length = 128`
- Hann window

Then compute **power spectrum** and apply **log scaling**.